In [ ]:
from langgraph.graph import StateGraph , START ,END
from langgraph.graph.message import add_messages
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace
from typing import TypedDict , Annotated 
from langchain_core.messages import BaseMessage , HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from dotenv import load_dotenv
load_dotenv()


In [ ]:
class chatbot_state(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]
    

In [ ]:
llm=HuggingFaceEndpoint(repo_id="mistralai/Mistral-7B-Instruct-v0.2",
                        task="chat-completion")
model=ChatHuggingFace(llm=llm)


def chat_node(state:chatbot_state):
    message=state["messages"]

    response=model.invoke(message)

    return {"messages":[response]}

In [ ]:
checkpointer=MemorySaver()
graph=StateGraph(chatbot_state)

graph.add_node("chat_node",chat_node)

graph.add_edge(START,"chat_node")
graph.add_edge("chat_node",END)

chat_bot=graph.compile(checkpointer=checkpointer)

In [ ]:
chat_bot

In [ ]:
query={"messages":[HumanMessage(content="who is their prime minister ")]}
chat_bot.invoke(query)

In [ ]:
thread_id="1"

while True:

    user_message=input("Enter your query")
    print("User Message:",user_message, "\n")

    if user_message.strip().lower() in ["quit","exit","end"]:
        break
    
    config={"configurable":{"thread_id":thread_id}}
    query={"messages":[HumanMessage(content=user_message)]}
    response=chat_bot.invoke(query,config)

    print(response["messages"][-1].content)
